# Reconocimiento de gestos en tiempo real

Este notebook usa el mejor checkpoint exportado en `outputs/models/gesture_recognition_(*).pt` para hacer inferencia con webcam mediante *OpenCV*. 
> No reentrena modelos; solo carga la arquitectura, los pesos y las clases guardadas.

## Pipeline

1. Capturar frames de la webcam.
2. Tomar una ROI fija donde se coloca la mano.
3. Preprocesar la ROI con uno de tres modos:
   - `raw`: escala de grises, resize a `92 x 92`, normalización `[0, 1]`. Es el modo por defecto y el más fiel al entrenamiento.
   - `dark-bg`: escala de grises con mano realzada y fondo oscurecido. Es un modo extra para acercar la webcam al estilo visual de LeapGestRecog.
   - `otsu`: escala de grises, desenfoque suave, umbralización de Otsu, resize a `92 x 92`, normalización `[0, 1]`. Es útil para comparar con el flujo descrito en el paper.
4. Ejecutar la CNN cargada desde el checkpoint.
5. Suavizar predicciones de varios frames y mostrar clase/confianza sobre el video.

In [2]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

PROJECT_ROOT

def short_path(p: Path) -> str:
    """Resume una ruta para mostrarla en salidas del notebook.

    Args:
        p: Ruta a resumir.

    Returns:
        Ruta abreviada con sus ultimos tres componentes.
    """
    return "..."+"/".join(p.parts[-3:])

In [3]:
from realtime_gesture_recognition import choose_model_path, display_model_key, load_checkpoint_model, resolve_device

device = resolve_device("auto")
model_path = choose_model_path(PROJECT_ROOT, explicit_model=None)
model, metadata = load_checkpoint_model(model_path, device)

print("Modelo:", short_path(model_path))
print("Arquitectura:", display_model_key(metadata["model_key"]))
print("Image size:", metadata["image_size"])
print("Dispositivo:", device)
print("Clases:")
for index, class_name in enumerate(metadata["class_names"]):
    print(f"{index}: {class_name}")

Modelo: ...outputs/models/gesture_recognition_(cnn_b).pt
Arquitectura: CNN-B
Image size: (92, 92)
Dispositivo: cuda
Clases:
0: palm
1: l
2: fist
3: fist_moved
4: thumb
5: index
6: ok
7: palm_moved
8: c
9: down


In [4]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "realtime_gesture_recognition.py"),
    "--smoke-test",
]
print(short_path(Path(" ".join(cmd))))
result = subprocess.run(cmd, check=True)
print("Proceso completado")

...Experimento_Final/scripts/realtime_gesture_recognition.py --smoke-test
Proceso completado


## Webcam

Coloca la mano dentro del cuadro verde, usa buena iluminación y un fondo simple. La ventana de OpenCV se abre fuera del notebook. Presiona `q` o `Esc` para salir.

In [8]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "realtime_gesture_recognition.py"),
    "--preprocess", "raw",
]
print(short_path(Path(" ".join(cmd))))
subprocess.run(cmd, check=True)
print("Proceso completado")

...Experimento_Final/scripts/realtime_gesture_recognition.py --preprocess raw
Proceso completado


In [ ]:
# Ejecuta esta celda si quieres comparar contra el modo extra dark-bg.
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "realtime_gesture_recognition.py"),
    "--preprocess", "dark-bg",
]
print(short_path(Path(" ".join(cmd))))
subprocess.run(cmd, check=True)
print("Proceso completado")

In [ ]:
# Ejecuta esta celda si quieres comparar contra el modo con umbralización de Otsu.
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "realtime_gesture_recognition.py"),
    "--preprocess", "otsu",
]
print(short_path(Path(" ".join(cmd))))
subprocess.run(cmd, check=True)
print("Proceso completado")